In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from datetime import datetime
import time
import pandas as pd
import re
from deep_translator import GoogleTranslator

In [ ]:
!where python

In [ ]:
# parse the dates

def parse_french_or_english_date(date_str):

    date_str = date_str.replace(",", "").strip()
    
    month_map = {
        "jan": "Jan", "janv": "Jan","Fév": "Feb", "fév": "Feb", "févr": "Feb", "fev": "Feb", "mars": "Mar", "avr": "Apr",
        "mai": "May", "juin": "Jun", "juil": "Jul", "août": "Aug", "aout": "Aug", "sept": "Sep",
        "oct": "Oct", "nov": "Nov", "déc": "Dec", "dec": "Dec",
        "janvier": "Jan", "février": "Feb", "fevrier": "Feb", "avril": "Apr", "juillet": "Jul",
        "septembre": "Sep", "octobre": "Oct", "novembre": "Nov", "décembre": "Dec", "decembre": "Dec",
        "jan": "Jan", "feb": "Feb", "mar": "Mar", "apr": "Apr", "may": "May", "jun": "Jun",
        "jul": "Jul", "aug": "Aug", "sep": "Sep", "oct": "Oct", "nov": "Nov", "dec": "Dec",
        "january": "Jan", "february": "Feb", "march": "Mar", "april": "Apr", "june": "Jun",
        "july": "Jul", "august": "Aug", "september": "Sep", "october": "Oct", "november": "Nov", "december": "Dec",
    }

    parts = date_str.lower().split()
    if len(parts) != 3:
        raise ValueError(f"Unexpected date format: {date_str}")

    month_raw, day, year = parts
    month_en = month_map.get(month_raw)
    
    if not month_en:
        raise ValueError(f"Parsing error '{date_str}': '{month_raw}'")
    new_date_str = f"{month_en} {day} {year}"

    return datetime.strptime(new_date_str, "%b %d %Y").date()


In [ ]:
# Notices, French version

driver = webdriver.Chrome()
driver.get("LINK") #add link
wait = WebDriverWait(driver, 10)

In [ ]:
# number of pages to scrape

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

try:
    page_info = soup.select_one("li.toppg-t").text.strip()

    total_pages = int(page_info.split("de")[-1].strip())
    print("Total number of pages:", total_pages)
except Exception as e:
    print("Error while extracting the number of pages:", e)

In [ ]:
# scraping

base_url = "https://appel-d-offre.dgmarket.com/tenders/list.do?sub=info-communications-in-Africa-10&locationISO=_s&language=fr&d-446978-s=p&d-446978-p=2&d-446978-n=1"
tenders=[]

for page in range(1, total_pages + 1):
    url = f"{base_url}&selPageNumber={page}"
    driver.get(url)

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".list_notice_table")))
    except:
        print("Timeout: content not loaded.")
        continue

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table", class_="list_notice_table")

    for table in tables:
        try:
            title_tag = table.select_one("div.ln_notice_title a")
            title = title_tag.text.strip()
            
            link = "https://www.dgmarket.com" + title_tag.get("href", "")

            country_tag = table.select_one("span.ln_listing a[href*='locationISO']")
            country = country_tag.text.strip() if country_tag else "Inconnu"

            date_tag = table.select_one("div.ln_date")
            raw_date = date_tag.text.strip()

            publish_date = None
            if raw_date:
                try:
                    date_obj = parse_french_or_english_date(raw_date)
                    if date_obj:
                        publish_date = date_obj.strftime('%d-%m-%Y')
                except Exception as e:
                    print(f"Date parsing error '{raw_date}': {e}")

            tenders.append({
                "title": title,
                "publication_date": publish_date,
                "expiration_date": None,
                "link": link,
                "country": country
            })

        except Exception as e:
            print(f"Error in a block: {e}")
            continue


In [ ]:
# translation

def translate_text(text, source='en', target='fr'):
    try:
        return GoogleTranslator(source=source, target=target).translate(text)
    except Exception as e:
        print("Translation error:", e)
        return text

In [ ]:
# Notices, English version

driver.get("https://appel-d-offre.dgmarket.com/tenders/list.do?sub=info-communications-in-Africa-10&locationISO=_s&language=en")

In [ ]:
# number of pages to scrape

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

try:
    page_info = soup.select_one("li.toppg-t").text.strip()

    total_pages_en = int(page_info.split("de")[-1].strip())
    print("Total number of pages:", total_pages_en)
except Exception as e:
    print("Error while extracting the number of pages:", e)

In [ ]:
# scraping and translation

base_url = "https://appel-d-offre.dgmarket.com/tenders/list.do?sub=info-communications-in-Africa-10&locationISO=_s&language=en&d-446978-s=p&d-446978-p=2&d-446978-n=1"

for page in range(1, total_pages_en + 1):
    url = f"{base_url}&selPageNumber={page}"
    driver.get(url)
    
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".list_notice_table")))
    
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table", class_="list_notice_table")

    for table in tables:
        try:
            title_tag = table.select_one("div.ln_notice_title a")
            title_en = title_tag.text.strip()
            title = translate_text(title_en)
            link = "https://www.dgmarket.com" + title_tag.get("href", "")

            country_tag = table.select_one("span.ln_listing a[href*='locationISO']")
            country = country_tag.text.strip() if country_tag else "Inconnu"

            date_tag = table.select_one("div.ln_date")
            raw_date = date_tag.text.strip()

            publish_date = None
            if raw_date:
                try:
                    date_obj = parse_french_or_english_date(raw_date)
                    if date_obj:
                        publish_date = date_obj.strftime('%d-%m-%Y')
                        print(publish_date)
                except Exception as e:
                    print(f"Date parsing error '{raw_date}': {e}")


            tenders.append({
                "title": title,
                "publication_date": publish_date,
                "expiration_date": None,
                "link": link,
                "country": country
            })

        except Exception as e:
            print(f"Error in a block: {e}")
            continue

In [ ]:
driver.quit()

In [ ]:
df = pd.DataFrame(tenders)
df.to_excel("../2_scraped_files/dgMarket.xlsx", index=False) # modify